# Sizing the quantizer2 Comparator and Dither

Adapted from `macros/ota_fcasc_fd/scripts/sizing/lookup_commands.ipynb`, which is the overview of the pygmid lookup commands this notebook uses.

**Copyright 2026 Michael Koefinger**

SPDX-License-Identifier: Apache-2.0 WITH SHL-2.1

quantizer2 was sized by hand and simulation, so this notebook rebuilds the derivation after the fact. It starts from the spec, derives what the gm/ID lookups and the regeneration, Ron and kT/C formulas can give, then reads the drawn sizes from the three schematics and compares. Nothing here changes a schematic. Where the derivation disagrees with the drawing, the cell says so and leaves the drawing alone.

Every number below is a gm/ID model estimate from the tt tables at 300 K, with no mismatch, no process corner and no layout wiring unless a cell says otherwise. The simulated values are in `macros/quantizer2/README.md` and are sanity checks. The tables hold no Pelgrom data, so offset is a target here and the Monte Carlo is its source.

The block is a StrongARM latch (input pair `N0`/`N1`, tail `N2`, cross-coupled `N3`/`N4` and `P0`/`P1`, reset PMOS `P2` to `P5`), the NOR SR latch and the library gates around it, and the dither: three auxiliary pairs in parallel with the input pair whose gates are held at the input plus or minus a ladder voltage by flying capacitors. A dynamic comparator has no bias point to set with gm/ID, so the knobs are the integration time, the regeneration time constant C/gm, kT/C and Ron. The library cells (inverters, nand2, nor2, and2) are not drawn in xschem, their widths come from the CDL export of the top schematic.

The 1st-order loop sets the requirements. The quantizer only needs a correct sign, so a large offset is harmless (see the offset cell). The decision has to be finished before the DAC select is used in phase 1, which is 3.24 ns after the quantizer clock rises at the worst corner (dsm2 README), not within the 488 ns half period.

In [1]:
# Packages
from pygmid import Lookup as lk
import numpy as np
import re
from pathlib import Path

In [2]:
# Spec and the choices the READMEs give
VDD, VCM = 1.5, 0.75
FS       = 0.75               # V, full scale around vdd/2 (dsm2 README)
FCLK     = 1.024e6            # Hz
T_HALF   = 0.5 / FCLK         # s, 488 ns
T_GAP    = 3.24e-9            # s, shortest qclk rise to phase 1 rise (dsm2 README, ff 1.55 V -40 C, clk_noverlap bench)
SLOW     = 1.41               # ss -40 C 1.45 V over tt, clock to latch (quantizer2 README, 1.218 ns over 0.865 ns)
V_RES    = 1e-9               # V, smallest input the comparator must still resolve inside the gap at the slow corner
VOS_3S   = 25e-3              # V, input offset (3 sigma) the loop tolerates, see the offset cell
A_DITH   = 75e-3              # V, nominal dither amplitude, code 7 at s[0]
A_WINDOW = (67e-3, 85e-3)     # V, window of the system model
STEPS    = {'s[3]': 1, 's[1]': 2, 's[0]': 3, 's[2]': 4}   # ladder taps from the centre, 33 / 67 / 100 / 133 %
RSQ      = 1476.0             # ohm per square, rhigh 0.5u wide, from the bias_gm README (2.17 MOhm for 1470 squares), not a table value
C_UM2    = 1.287              # fF per um2, cap_cmomf model
CAP_REAL = 0.70               # estimated real over model of the flying cap unit, taken from the sc_int2 values (0.73 and 0.67), no field solve for this shape
C_PAD    = 80e-15             # F, load of the pad driver benches
SIM = dict(                   # quantizer2 README, tt 27 C 1.5 V, schematic, sanity checks
    t_rise = {10e-3: 0.859e-9, 1e-3: 0.983e-9, 1e-6: 1.639e-9},
    t_fall = {10e-3: 0.792e-9, 1e-3: 0.911e-9, 1e-6: 1.065e-9},
    A = 76.5e-3, i_ladder = 2.6e-6, q_pin = (45.9e-15, 75.3e-15), t_pad = 1.053e-9 - 0.885e-9, d_gate = 26e-12)
K_B_T    = 1.380649e-23 * 300
CELLS    = ('quantizer2_sal', 'quantizer2_dith', 'quantizer2_top')
DATA     = Path('../../../macros/inverter/scripts/sizing/data')
SCHD     = Path('../schematic')
CDL      = Path('netlist/quantizer2_top_klayout.cdl')

### Read the sizes out of the schematics

The schematics are the source of truth, so nothing here restates a drawn size. `read_sizes` returns the total W (the `ng` fingers are inside it), L and the instance count of every device, vectors such as `MN43[3:0]` counted by their range. The checks below are the invariants the topology rests on: one channel length for the comparator, matched pairs, the tail at twice an input pair, the aux pairs at 1 : 2 : 4 and the flying capacitors at 1 : 2 : 4 units.

In [3]:
def num(s):
    # 500n, 0.13u, 4.1e-6 or a plain number in m, all returned in um
    if s.endswith('n'): return float(s[:-1]) * 1e-3
    if s.endswith('u'): return float(s[:-1])
    return float(s) * 1e6

def count(name):
    # a vector instance such as MN43[3:0] is one device per index
    m = re.search(r'\[(\d+):(\d+)\]', name)
    return abs(int(m.group(1)) - int(m.group(2))) + 1 if m else 1

def read_sizes(path):
    txt = path.read_text()
    dev = {}
    for m in re.finditer(r'^C \{([^}]+)\.sym\} \S+ \S+ \S+ \S+ \{name=(\S+)\n(.*?)\}', txt, re.S | re.M):
        a = dict(re.findall(r'^(\w+)=(\S+)$', m.group(3), re.M))
        if 'w' in a and 'l' in a:
            dev[m.group(2)] = dict(w=num(a['w']), l=num(a['l']), n=count(m.group(2)), sym=m.group(1))
    return dev

def read_cells(path):
    # the library cells of the top schematic as exported in its CDL: (type, d, g, s, W, L) per MOS
    cells, cur = {}, None
    for ln in path.read_text().splitlines():
        m = re.match(r'\.subckt (sg13cmos5l_\w+) ', ln)
        if m: cur = m.group(1); cells[cur] = []; continue
        m = re.match(r'M\w+ (\S+) (\S+) (\S+) \S+ sg13_lv_([np])mos w=(\S+) l=(\S+)', ln)
        if m and cur: cells[cur].append((m.group(4), m.group(1), m.group(2), m.group(3), num(m.group(5)), num(m.group(6))))
        if ln.startswith('.ends'): cur = None
    return cells

D = {c: read_sizes(SCHD / (c + '.sch')) for c in CELLS}
S, DT, TOP = D['quantizer2_sal'], D['quantizer2_dith'], D['quantizer2_top']
STD = read_cells(CDL)
LATCH = ('N0', 'N1', 'N2', 'N3', 'N4', 'P0', 'P1', 'P2', 'P3', 'P4', 'P5')
for n in LATCH:
    print(f"{n:3s} W {S[n]['w']:5.1f} um  L {S[n]['l']:.2f} um")
print('aux pairs W (um), q side and p side:', [S[k]['w'] for k in ('MAQ0', 'MAQ1', 'MAQ2')], [S[k]['w'] for k in ('MAP0', 'MAP1', 'MAP2')])
NCAP = {b: sum(DT[k]['n'] for k in DT if re.fullmatch(f'CFQ{b}(\\[\\d+:\\d+\\])?', k)) for b in '012'}
print('flying cap units per side and bit:', NCAP, ', dummy', DT['CFQD']['n'], ', unit', DT['CFQ0']['w'], 'x', DT['CFQ0']['l'], 'um')

# invariants the topology rests on
assert {S[n]['l'] for n in LATCH} == {0.5}, 'mixed comparator lengths'
assert S['N0']['w'] == S['N1']['w'] and S['N3']['w'] == S['N4']['w'] and S['P0']['w'] == S['P1']['w'], 'pair mismatch'
assert S['N2']['w'] == 2 * S['N1']['w'], 'tail is not twice an input'
WA = [S[k]['w'] for k in ('MAQ0', 'MAQ1', 'MAQ2')]
assert WA == [S[k]['w'] for k in ('MAP0', 'MAP1', 'MAP2')] and WA[1] == 2 * WA[0] and WA[2] == 4 * WA[0], 'aux pairs are not 1:2:4'
assert list(NCAP.values()) == [1, 2, 4], NCAP

N0  W  16.0 um  L 0.50 um
N1  W  16.0 um  L 0.50 um
N2  W  32.0 um  L 0.50 um
N3  W   2.0 um  L 0.50 um
N4  W   2.0 um  L 0.50 um
P0  W   6.0 um  L 0.50 um
P1  W   6.0 um  L 0.50 um
P2  W   6.0 um  L 0.50 um
P3  W   6.0 um  L 0.50 um
P4  W   6.0 um  L 0.50 um
P5  W   6.0 um  L 0.50 um
aux pairs W (um), q side and p side: [1.5, 3.0, 6.0] [1.5, 3.0, 6.0]
flying cap units per side and bit: {'0': 1, '1': 2, '2': 4} , dummy 1 , unit 7.6 x 4.1 um


### Lookups

Extensive quantities come from the tables, which are swept at W = 5 um, and are scaled to the drawn width. The tables have L = 0.13 um, used for the library cells and the switches, and L = 0.5 um for the comparator. The on resistance of a switching device is taken as VDD/2 over the drain current at full gate drive, which is the large-signal value that a delay of 0.69 R C needs, not the small-signal triode value.

In [4]:
nm = lk(str(DATA / 'sg13cmos5l_lv_nmos.mat'))
pm = lk(str(DATA / 'sg13cmos5l_lv_pmos.mat'))
T = {'n': nm, 'p': pm}

def ext(t, f, L, vgs, vds, vsb=0.0, w=1.0):
    # extensive quantity of a device of width w (um), the table is swept at its own W
    return float(t.lookup(f, L=L, VGS=vgs, VDS=vds, VSB=vsb)) / float(t['W']) * w

def inten(t, f, L, vgs, vds, vsb=0.0):
    return float(t.lookup(f, L=L, VGS=vgs, VDS=vds, VSB=vsb))

def ron_eff(kind, L, w):
    # large-signal on resistance of a switching device: VDD/2 over ID at full gate drive
    return VDD / 2 / ext(T[kind], 'ID', L, VDD, VDD / 2, 0.0, w)

def cgg(kind, L, w, vgs=0.75):
    return abs(ext(T[kind], 'CGG', L, vgs, 0.0, 0.0, w))

def cdd(kind, L, w, vgs=0.0, vds=0.75):
    return abs(ext(T[kind], 'CDD', L, vgs, vds, 0.0, w))

def cin(cell, pin):
    return sum(cgg(t, l, w) for t, d, g, s, w, l in STD[cell] if g == pin)

def cout(cell, pin):
    return sum(cdd(t, l, w) for t, d, g, s, w, l in STD[cell] if pin in (d, s) and pin not in ('VDD', 'VSS'))

### Operating point of the input pair

During the evaluation the tail is in triode at VGS = VDD and the pair sits at the input common mode of 0.75 V. The tail node `vtail` solves tail current equal to twice the current of one input device, with the drain still at VDD. The drawn tail is twice the pair in width so that the tail, deep in triode, does not limit the pair.

The tail carries 1.78 mA, but only for the first nanosecond of every clock, so the average is about 1 uA. One input device has gm = 4.05 mS at gm/ID = 4.5 1/V, which is strong inversion on purpose: the gain of the pre-amplification is gm times a few tens of picoseconds over the node capacitance, and a high gm/ID would only cut current that flows for a nanosecond anyway.

In [5]:
L = 0.5
WIN, WTAIL = S['N1']['w'] * S['N1']['n'], S['N2']['w'] * S['N2']['n']
WNL, WPL = S['N4']['w'], S['P1']['w']

def solve_tail(win, wtail, vcm=VCM):
    # tail in triode at VGS = VDD, the pair carries half the tail current each, VDS of the pair counted from the precharged drain
    lo, hi = 0.0, 0.7
    for _ in range(60):
        vt = (lo + hi) / 2
        f = ext(nm, 'ID', L, VDD, vt, 0.0, wtail) - 2 * ext(nm, 'ID', L, vcm - vt, VDD - vt, vt, win)
        lo, hi = (lo, vt) if f > 0 else (vt, hi)
    return vt

def optab(win, wtail):
    vt = solve_tail(win, wtail)
    it = ext(nm, 'ID', L, VDD, vt, 0.0, wtail)
    gm = ext(nm, 'GM', L, VCM - vt, 1.0, vt, win)
    return vt, it, gm

VT, ITAIL, GM_IN = optab(WIN, WTAIL)
VTN = inten(nm, 'VTH', L, 0.5, 0.5, 0.7)        # N4, N3 turn on with their sources at about VDD/2
print(f'vtail {VT*1e3:.0f} mV, tail current {ITAIL*1e3:.2f} mA, gm of one input {GM_IN*1e3:.2f} mS, gm/ID {GM_IN/(ITAIL/2):.1f} 1/V, VTH of N4 {VTN*1e3:.0f} mV')

vtail 78 mV, tail current 1.78 mA, gm of one input 4.05 mS, gm/ID 4.5 1/V, VTH of N4 342 mV


### Integration and regeneration

After the clock rises the tail pulls `v_p` and `v_q` down from VDD. The cross-coupled NMOS `N3`/`N4` conduct once their sources have fallen by one VTH, which takes T1 = CX VTH / (Itail/2). The input difference at that moment is gm vin T1 / CX. From there the latch regenerates with tau = CL / (gm_n + gm_p), where CL is the capacitance on `do_p` (four drains, the two latch gates, the dummies and the `inv_1` of the SR path) and gm is that of the two cross-coupled devices at mid rail.

The model gives CX = 34.8 fF, CL = 69.5 fF and tau = 74 ps. The simulated rise times at 1 mV and 1 uV overdrive imply 95 ps. The falling edge implies only 22 ps because it goes through the other side of the SR latch and does not follow the regeneration. The model is 22 % below the rising edge, which is the direction a model that leaves out the wiring should be.

In [6]:
# v_p: drains of N1, P5 and the aux pairs of that side, source of N4
AUX_P = sum(S[k]['w'] * S[k]['n'] for k in ('MAP0', 'MAP1', 'MAP2', 'MAPD'))
CX = (cdd('n', L, WIN, 0.6, 1.0) + cdd('p', L, S['P5']['w']) + cdd('n', L, AUX_P, 0.0, 1.0)
      + abs(ext(nm, 'CSS', L, 0.6, 0.5, 0.0, S['N4']['w'])))
# do_p: drains of N4, P1, P3 and the dummies, gates of N3 and P0, the inv_1 of the SR path
NB = S['MN43[3:0]']
CL = (cdd('n', L, S['N4']['w']) + cdd('p', L, S['P1']['w']) + cdd('p', L, S['P3']['w'])
      + cdd('n', L, NB['w'] * NB['n']) + cdd('p', L, S['MPDUM1[3:0]']['w'] * S['MPDUM1[3:0]']['n'])
      + cgg('n', L, S['N3']['w']) + cgg('p', L, S['P0']['w']) + cin('sg13cmos5l_inv_1', 'A'))
GM_L = ext(nm, 'GM', L, VCM, VCM, 0.0, WNL) + ext(pm, 'GM', L, VCM, VCM, 0.0, WPL)
TAU = CL / GM_L
T1 = CX * VTN / (ITAIL / 2)
G_PRE = GM_IN / GM_L
print(f'CX {CX*1e15:.1f} fF, CL {CL*1e15:.1f} fF, gm of the latch pairs {GM_L*1e3:.2f} mS')
print(f'regeneration tau = CL / gm = {TAU*1e12:.0f} ps, N4 turns on after {T1*1e12:.0f} ps')
sim = SIM['t_rise']
print(f"tau from the simulated 1 mV to 1 uV rise times: {(sim[1e-6]-sim[1e-3])/np.log(1e3)*1e12:.0f} ps (falling edge {(SIM['t_fall'][1e-6]-SIM['t_fall'][1e-3])/np.log(1e3)*1e12:.0f} ps)")

CX 34.8 fF, CL 69.5 fF, gm of the latch pairs 0.94 mS
regeneration tau = CL / gm = 74 ps, N4 turns on after 13 ps
tau from the simulated 1 mV to 1 uV rise times: 95 ps (falling edge 22 ps)


### Clock path and output chain

Outside the latch two paths add to the clock to `do_qi` time. The `and2_2` that gates the clock drives the tail and the four reset PMOS (353 fF of gate), and the signal goes from `do_p` through `inv_1`, `inv_2`, the preset gate and the SR NOR. Each stage is 0.69 R C with R the mean of the pull-up and pull-down large-signal Ron (series devices counted) and C the next input plus the own drain capacitance.

In [7]:
def stage(cell, load, stack_n=1, stack_p=1):
    # 0.69 R C of a library gate, mean of the rising and falling output
    devs = STD[cell]
    wn = sum(w for t, d, g, s, w, l in devs if t == 'n' and d in ('Y', 'X'))
    wp = sum(w for t, d, g, s, w, l in devs if t == 'p' and d in ('Y', 'X'))
    r = (ron_eff('n', 0.13, wn) * stack_n + ron_eff('p', 0.13, wp) * stack_p) / 2
    return 0.69 * r * (load + sum(cdd(t, l, w) for t, d, g, s, w, l in devs if d in ('Y', 'X')))

c_inv1, c_inv2 = cin('sg13cmos5l_inv_1', 'A'), cin('sg13cmos5l_inv_2', 'A')
c_nand, c_nor  = cin('sg13cmos5l_nand2_2', 'A'), cin('sg13cmos5l_nor2_2', 'A')
c_sr_load = c_nor + cin('sg13cmos5l_and2_1', 'A') + 2 * c_inv1        # nor2_2 other input, pad and2, dec inverter
t_chain = (stage('sg13cmos5l_inv_1', c_inv2) + stage('sg13cmos5l_inv_2', c_nand)
           + stage('sg13cmos5l_nand2_2', c_nor, 2, 1) + stage('sg13cmos5l_nor2_2', c_sr_load, 1, 2))
# the clock and2_2 drives the gates of the tail and the four reset PMOS
c_gclk = cgg('n', L, WTAIL) + cgg('p', L, sum(S[k]['w'] for k in ('P2', 'P3', 'P4', 'P5'))) + cgg('p', L, S['P2']['w']) * 0
t_clk = stage('sg13cmos5l_and2_2', c_gclk)
print(f'clock path: and2_2 into {c_gclk*1e15:.0f} fF of gclk gates {t_clk*1e12:.0f} ps')
print(f'latch to do_qi: inv_1, inv_2, preset nand2_2 and SR nor2_2 {t_chain*1e12:.0f} ps')

clock path: and2_2 into 353 fF of gclk gates 235 ps
latch to do_qi: inv_1, inv_2, preset nand2_2 and SR nor2_2 43 ps


### Decision time against the simulation

The model is 0.24 to 0.38 ns below the simulated rising edge at all three overdrives and 0.51 ns against 0.66 ns for the span between 1 mV and 1 uV, so it is a lower bound about 30 % short. What it leaves out is the slow input edge of the clock, the turn-on of the 32u tail and the wiring. The structure is right: the decision time is a fixed part of about 0.6 ns plus tau times the logarithm of the overdrive.

In [8]:
def t_dec(vin, g_in=GM_IN, cx=CX, itail=ITAIL, tau=TAU, extra=t_clk + t_chain):
    t1 = cx * VTN / (itail / 2)
    dv = g_in * vin * t1 / cx
    return t1 + tau * np.log(VDD / 2 / dv) + extra

print(' overdrive    model   sim rising   sim falling   (ns)')
for vin in (10e-3, 1e-3, 1e-6):
    print(f"{vin:9.0e} {t_dec(vin)*1e9:8.3f} {SIM['t_rise'][vin]*1e9:10.3f} {SIM['t_fall'][vin]*1e9:12.3f}")

 overdrive    model   sim rising   sim falling   (ns)
    1e-02    0.578      0.859        0.792
    1e-03    0.748      0.983        0.911
    1e-06    1.257      1.639        1.065


### Metastability margin

A late decision is the failure mode of the latch. The requirement is that a 1 nV input is still resolved inside the 3.24 ns gap at the slow corner, which the model does in 2.49 ns, 0.75 ns to spare. The smallest input that resolves in the gap is then 0.75 pV. The input of the comparator in the loop has an rms of about 0.3 V (the integrator output swings between -0.39 and +0.5 V), so the chance that a clock lands in that window is 1e-12 and an event comes once in about 1e6 s. An input that close to zero has no wrong answer to give, so what the event can disturb is only the level of the SR output, which is why the requirement is this loose. The README gives 1.731 ns at 1 uV overdrive at ss -40 C 1.45 V, 1.87 times inside the gap, which agrees with the model at the 1 uV point.

In [9]:
t_slow = SLOW * t_dec(V_RES)
budget = T_GAP
dv_min = lambda t_av: VDD / 2 * np.exp(-(t_av - T1 - t_clk - t_chain) / TAU) * CX / (GM_IN * T1)   # smallest input still resolved in t_av
print(f'{V_RES*1e9:.0f} nV decision, slow corner {t_slow*1e9:.2f} ns against the {T_GAP*1e9:.2f} ns gap')
print(f'smallest input resolved inside the gap (slow corner tau): {dv_min(T_GAP/SLOW):.1e} V')
VID_RMS = 0.3          # V, rms of the quantizer input in the loop (dsm2 README, vo_p - vo_n between -0.39 and +0.5 V)
p_meta = dv_min(T_GAP/SLOW) / (np.sqrt(2*np.pi) * VID_RMS)       # probability density of the input at 0 times the window
print(f'chance that a clock lands in the window {p_meta:.1e}, one per {1/(p_meta*FCLK):.1e} s')

1 nV decision, slow corner 2.49 ns against the 3.24 ns gap
smallest input resolved inside the gap (slow corner tau): 7.5e-13 V
chance that a clock lands in the window 1.0e-12, one per 9.8e+05 s


### What sets the widths

Speed and noise do not set the input pair. The decision time is flat in W because the tail current, CX and gm all scale together (2.4 to 2.7 ns at 1 nV and the slow corner from 2u to 32u), and the input referred noise is 0.57 mV at 2u and 0.16 mV at 16u against a limit of 3.6 mV, a sixth of the dither level spacing. So the sweep ends at its smallest entry. What grows with the width is the kickback (the gate capacitance, 12.7 fF at 2u and 102 fF at 16u) and what falls is the offset, so the drawn 16u is an offset and layout choice that this notebook cannot derive, the Monte Carlo can.

The cross-coupled pairs behave the same way: tau is 103 ps at a quarter of the drawn width and 69 ps at twice, because the load is mostly their own capacitance. The reset PMOS settle `v_p` in 55 ps and `do_p` in 110 ps at 6u, and a 0.5u device would still settle in a twentieth of the half period, so their 6u is set by symmetry with the latch PMOS, not by speed.

In [10]:
SIGMA_MAX = 2 * A_DITH / 7 / 6      # V, noise a sixth of the dither level spacing
print(f'noise limit {SIGMA_MAX*1e3:.1f} mV rms')
print(' Win/um  Wtail/um  vtail/mV  Itail/mA  t(1nV) slow/ns  sigma_in/mV  Cgg(Win)/fF')
rows = {}
for win in (2, 4, 8, 16, 24, 32):
    vt, it, gm = optab(win, 2 * win)
    cx = CX - cdd('n', L, WIN, 0.6, 1.0) + cdd('n', L, win, 0.6, 1.0)
    cg = c_gclk - cgg('n', L, WTAIL) + cgg('n', L, 2 * win)
    ts = SLOW * t_dec(V_RES, gm, cx, it, TAU, stage('sg13cmos5l_and2_2', cg) + t_chain)
    sig = np.sqrt(K_B_T / CL / (gm / GM_L) ** 2 + 4 * K_B_T / gm * (1 / (2 * (cx * VTN / (it / 2) + TAU))))
    rows[win] = (ts, sig)
    print(f'{win:6d} {2*win:9d} {vt*1e3:9.0f} {it*1e3:9.2f} {ts*1e9:14.2f} {sig*1e3:12.2f} {cgg("n", L, win)*1e15:11.1f}')
W_MIN = min(w for w, (ts, sg) in rows.items() if ts <= budget and sg <= SIGMA_MAX)
print(f'smallest input pair on this list that meets both: {W_MIN} um')

# latch pairs scaled together: the fixed load is the inv_1 input and the dummies
c_fix = cin('sg13cmos5l_inv_1', 'A') + cdd('n', L, NB['w'] * NB['n']) + cdd('p', L, S['MPDUM1[3:0]']['w'] * S['MPDUM1[3:0]']['n'])
print('latch scale  tau/ps  (width of N3, N4 / P0, P1 in um)')
for sc in (0.25, 0.5, 1, 2):
    print(f'{sc:10.2f} {(CL - c_fix) * sc / (GM_L * sc) * 1e12 + c_fix / (GM_L * sc) * 1e12:7.0f}  ({WNL*sc:g} / {WPL*sc:g})')

# reset PMOS: Ron * C must settle the node in a tenth of a half period, five time constants
for nm_, c_node in (('v_p / v_q (P4, P5)', CX), ('do_p / do_n (P2, P3)', CL)):
    w_min = 0.5
    while 5 * ron_eff('p', L, w_min) * c_node > T_HALF / 10: w_min += 0.5
    print(f'{nm_:22s} Ron*C with the drawn {S["P5"]["w"]:g}u: {ron_eff("p", L, S["P5"]["w"])*c_node*1e12:5.1f} ps, smallest W for 5 tau in T/20: {w_min:g} um')

noise limit 3.6 mV rms
 Win/um  Wtail/um  vtail/mV  Itail/mA  t(1nV) slow/ns  sigma_in/mV  Cgg(Win)/fF
     2         4        78      0.22           2.40         0.57        12.7
     4         8        78      0.45           2.38         0.35        25.4


     8        16        78      0.89           2.41         0.24        50.8
    16        32        78      1.78           2.49         0.16       101.6
    24        48        78      2.68           2.58         0.13       152.4
    32        64        78      3.57           2.68         0.11       203.2
smallest input pair on this list that meets both: 2 um


latch scale  tau/ps  (width of N3, N4 / P0, P1 in um)
      0.25     103  (0.5 / 1.5)
      0.50      84  (1 / 3)
      1.00      74  (2 / 6)
      2.00      69  (4 / 12)
v_p / v_q (P4, P5)     Ron*C with the drawn 6u:  55.3 ps, smallest W for 5 tau in T/20: 0.5 um
do_p / do_n (P2, P3)   Ron*C with the drawn 6u: 110.4 ps, smallest W for 5 tau in T/20: 0.5 um


### Offset target

A 1st-order loop with a 1 bit quantizer tolerates a large input offset. The offset adds to the quantizer input and enters the loop like the quantization error, so it is shaped by the same (1 - z^-1). Its DC part is removed by the integrator, the average of the output bits does not move. What it costs is integrator headroom, 25 mV against the 1.42 V swing of the OTA is under 4 %, and a shift of the centre of the dither levels. The integrator leak of 3.5e-4 per cycle (sc_int2 README, sim) turns 25 mV into about 9 uV at the input, 0.001 % of the 0.75 V full scale.

The target is therefore 3 sigma of 25 mV, a third of the dither amplitude and 1.2 code steps. The tables hold no Pelgrom data. The cell lists how the pairs weigh in, with sigma of VT taken as 1/sqrt(W L) and the latch pairs divided by the pre-gain gm_in/(gm_n + gm_p) of 4.3. The input pair dominates at 8 um2 per device, the latch NMOS at 1 um2 adds 0.66 of it. The Monte Carlo is the source for the actual sigma.

In [11]:
print(f'input pair area W*L = {WIN*L:.1f} um2 per device, latch NMOS {WNL*L:.1f}, latch PMOS {WPL*L:.1f}')
print(f'gain from the input to the latch, gm_in / (gm_n + gm_p) = {G_PRE:.1f}')
A_IN = WIN * L
print('relative sigma of each pair at the input, input pair = 1 (sigma_VT scales with 1 / sqrt(W L), pre-gain divides the latch pairs):')
rel = {'input pair': 1.0, 'latch NMOS': np.sqrt(A_IN / (WNL * L)) / G_PRE, 'latch PMOS': np.sqrt(A_IN / (WPL * L)) / G_PRE}
for k, v in rel.items():
    print(f'  {k:11s} {v:.2f}')
print(f'  sum in quadrature, two devices each: {np.sqrt(sum(v**2 for v in rel.values())):.2f} x the input pair alone')
print(f'target: 3 sigma <= {VOS_3S*1e3:.0f} mV, so sigma <= {VOS_3S/3*1e3:.1f} mV, to be checked by the Monte Carlo')
print(f'offset against the dither: 3 sigma is {VOS_3S/A_DITH:.2f} of the amplitude A and {VOS_3S/(2*A_DITH/7):.2f} of one code step')

input pair area W*L = 8.0 um2 per device, latch NMOS 1.0, latch PMOS 3.0
gain from the input to the latch, gm_in / (gm_n + gm_p) = 4.3
relative sigma of each pair at the input, input pair = 1 (sigma_VT scales with 1 / sqrt(W L), pre-gain divides the latch pairs):
  input pair  1.00
  latch NMOS  0.66
  latch PMOS  0.38
  sum in quadrature, two devices each: 1.26 x the input pair alone
target: 3 sigma <= 25 mV, so sigma <= 8.3 mV, to be checked by the Monte Carlo
offset against the dither: 3 sigma is 0.33 of the amplitude A and 1.17 of one code step


### Dither

The ladder is `RTOP` and `RBOT` of 80.4 um and eight `RL` of 4.08 um in rhigh, 0.5 um wide. With 1476 ohm per square (not a table value, taken from the bias_gm README) that is 571 kOhm and 2.63 uA, against the 2.6 uA of the README, and 31.6 mV per tap step. The four amplitude selects sit 1, 2, 3 and 4 steps from the centre, which is 33, 67, 100 and 133 %.

The aux pairs add to the input pair in parallel. The decision balances gm_main vid + gm_aux (alpha vid + 2 delta) = 0, so the amplitude is A = 2 delta x / (1 + alpha x) with x the aux over main transconductance and alpha the share of vi that the floating aux gate follows. With alpha = 1 and the width ratio x = 0.656 the model gives 75.2 mV at `s[0]` against 76.5 mV simulated, and 33.4, 66.6 and 133.4 % for the other selects against 30.5, 65.5 and 136.5 % in the README. The aux width that gives 75 mV at this ladder is 10.5 um, the drawn total. The 75 mV is the design target of the README, so this row agrees by construction.

One check for the planned dither-loss investigation. The code levels are A (2c - 7)/7, rms 49 mV. If the dither adds to the quantizer input it is shaped like the quantization error, and its variance is 2.4e-3 V2 against 0.25^2 / 3 = 0.0208 V2 of the 1 bit quantization error on the integrator output scale (DAC step 0.25 V), a ratio of 0.12 or about 0.5 dB. The first version of this cell compared against the error on a 1.5 V step and said 0.06 dB, which was wrong. The 7.2 dB of the single dither-on run turned out to be the LFSR window, see the dsm2 README.

In [12]:
R_LAD = RSQ * (DT['RTOP']['l'] + DT['RBOT']['l'] + DT['RL[7:0]']['n'] * DT['RL[7:0]']['l']) / DT['RTOP']['w']
I_LAD = VDD / R_LAD
DV_STEP = I_LAD * RSQ * DT['RL[7:0]']['l'] / DT['RL[7:0]']['w']
WAUX = sum(WA)
print(f'ladder {R_LAD/1e3:.0f} kOhm, {I_LAD*1e6:.2f} uA (sim {SIM["i_ladder"]*1e6:.1f} uA), {DV_STEP*1e3:.1f} mV per tap step')

# gm of the aux pairs against the main pair at the same bias, gates 44 mV above the input common mode (quantizer2 README)
gm_u = lambda vg: ext(nm, 'GM', L, vg - VT, 1.0, VT, 1.0)
x = WAUX / WIN
x_gm = gm_u(VCM + 0.044) / gm_u(VCM) * x
print(f'aux over main: W ratio {x:.3f}, gm ratio at the drawn gate level {x_gm:.3f}')

def amp(delta, x, alpha=1.0):
    # balance of gm_main*vid + gm_aux*(alpha*vid + 2 delta) = 0
    return 2 * delta * x / (1 + alpha * x)

print('amplitude A (code 7) per amplitude select, alpha = 1:')
for k, n in STEPS.items():
    print(f'  {k}: tap {n*DV_STEP*1e3:6.1f} mV, A {amp(n*DV_STEP, x)*1e3:6.1f} mV, {n/3*100:.0f} % of s[0]')
# the dither adds to the quantizer input and is shaped like the quantization error, so compare variances
LEV = [A_DITH * (2 * c - 7) / 7 for c in range(8)]
v_d, v_q = np.mean(np.square(LEV)), (2 * FS) ** 2 / 12
print(f'dither rms {np.sqrt(v_d)*1e3:.1f} mV, variance {v_d:.2e} V2 against the 1 bit quantization error {v_q:.3f} V2 ({FS*2:.1f} V step): {10*np.log10(1+v_d/v_q):.2f} dB more shaped noise')
print(f'window {A_WINDOW[0]*1e3:.0f} to {A_WINDOW[1]*1e3:.0f} mV, sim at tt {SIM["A"]*1e3:.1f} mV, level spacing 2A/7 = {2*amp(3*DV_STEP, x)/7*1e3:.1f} mV')
x_need = (A_DITH / (2 * 3 * DV_STEP)) / (1 - A_DITH / (2 * 3 * DV_STEP))
print(f'aux width that gives A = {A_DITH*1e3:.0f} mV at this ladder: {x_need*WIN:.1f} um in total, drawn {WAUX:.1f} um')

ladder 571 kOhm, 2.63 uA (sim 2.6 uA), 31.6 mV per tap step


aux over main: W ratio 0.656, gm ratio at the drawn gate level 0.722
amplitude A (code 7) per amplitude select, alpha = 1:
  s[3]: tap   31.6 mV, A   25.1 mV, 33 % of s[0]
  s[1]: tap   63.3 mV, A   50.1 mV, 67 % of s[0]
  s[0]: tap   94.9 mV, A   75.2 mV, 100 % of s[0]
  s[2]: tap  126.6 mV, A  100.3 mV, 133 % of s[0]
dither rms 49.1 mV, variance 2.41e-03 V2 against the 1 bit quantization error 0.188 V2 (1.5 V step): 0.06 dB more shaped noise
window 67 to 85 mV, sim at tt 76.5 mV, level spacing 2A/7 = 21.5 mV
aux width that gives A = 75 mV at this ladder: 10.5 um in total, drawn 10.5 um


### Flying capacitors and what the plates load

The floating aux gate follows vi through a divider of the flying capacitor and the aux gate capacitance, alpha = Cf / (Cf + Cgg). Both scale 1 : 2 : 4 over the three bits, so alpha is the same for all of them, 0.843 with the model capacitance and 0.790 with the estimated real one (0.70 of the model, assumed from the sc_int2 values 0.73 and 0.67, there is no field solve for this shape). With the lookup alpha and the gm ratio of 0.722 at the drawn gate level the model gives 85.2 mV at `s[0]`, 11 % above the simulation. The simulation sits between alpha = 1 and the lookup, so the aux gate capacitance in the simulation is lower than the table value or there is more loss than the model has, and this notebook cannot tell which. On silicon the smaller real Cf raises the model amplitude by 2.3 % (85.2 to 87.2 mV), small, but towards the upper edge of the 67 to 85 mV window. The ss -40 C corner at 70.0 mV in the README is 3 mV above the lower edge.

The kT/C of the smallest held gate is 0.38 mV, 0.32 mV at the input, against a code spacing of 21.5 mV. Floating the gate through the vtail jump of about 0.8 V at the falling clock moves it by 23 mV through Cgs, 25 % of the 100 % tap, which is the order of the 40 % amplitude error the README saw with a 1 ns delay and why the gates are held on the ladder through the jump.

When a plate moves from the ladder centre to vi, the floating gate puts Cf in series with its own Cgg. The charge on that series capacitance (44 fF over the three bits) comes out of the integrator output, and at the next high phase it goes to the ladder centre. For the integrator output this is a switched-capacitor resistor to the ladder centre of 22 MOhm per side (45 nS at 1.024 MHz). It exists only with the dither on. The OTA output conductance gm/A0 comes from the open loop gain of 58.84 dB and the unity gain frequency of 11.7 MHz of `ota_fcasc2` with its effective load, which is not known here, and is 13 nS at 0.15 pF, 25 nS at 0.3 pF and 50 nS at 0.6 pF. So the dither network may add about as much output conductance as the OTA has itself and cut the integrator DC gain by a factor of 1.9 to 4.5. The charge per pin and cycle at a plate step of 0.5 V is 22 fC, the README has 29.4 fC more with the dither on, and the input gate area ratio of 1.66 matches the simulated charge ratio of 1.64.

In [13]:
CF_MODEL = DT['CFQ0']['w'] * DT['CFQ0']['l'] * C_UM2 * 1e-15
print(f'flying cap unit {CF_MODEL*1e15:.1f} fF model, {CF_MODEL*CAP_REAL*1e15:.1f} fF estimated real')
print('bit  Cf model  Cf real   Cgg aux   alpha model   alpha real')
for b, (wb, n) in enumerate(zip(WA, NCAP.values())):
    cg = abs(ext(nm, 'CGG', L, VCM + 0.044 - VT, 0.5, VT, wb))
    am, ar = (n * CF_MODEL) / (n * CF_MODEL + cg), (n * CF_MODEL * CAP_REAL) / (n * CF_MODEL * CAP_REAL + cg)
    print(f'{b:3d} {n*CF_MODEL*1e15:9.1f} {n*CF_MODEL*CAP_REAL*1e15:8.1f} {cg*1e15:9.1f} {am:12.3f} {ar:12.3f}')
ALPHA_M, ALPHA_R = am, ar
print(f'A at s[0] with alpha model {amp(3*DV_STEP, x_gm, ALPHA_M)*1e3:.1f} mV, with alpha real {amp(3*DV_STEP, x_gm, ALPHA_R)*1e3:.1f} mV, sim {SIM["A"]*1e3:.1f} mV')

# kT/C of the held gate, smallest bit
kt = np.sqrt(K_B_T / (NCAP['0'] * CF_MODEL * CAP_REAL))
print(f'kT/C of the held gate of bit 0: {kt*1e3:.2f} mV rms, referred to the input {kt*x_gm/(1+x_gm)*2*1e3:.2f} mV')

# charge the plates take from vi: the floating gate puts Cf in series with its own Cgg, and the charge goes to the ladder centre at the next high phase
c_ser = sum((n * CF_MODEL) * cg / (n * CF_MODEL + cg) for n, cg in zip(NCAP.values(), [abs(ext(nm, 'CGG', L, VCM + 0.044 - VT, 0.5, VT, w)) for w in WA]))
g_sc = FCLK * c_ser
UGB, A0 = 11.7e6, 10 ** (58.84 / 20)      # ota_fcasc2 README, open loop, 58.84 dB and 11.7 MHz with CL + (1 - beta) C2 as load
print(f'series capacitance on each vi pin {c_ser*1e15:.1f} fF, a switched-capacitor resistor of {1/g_sc/1e6:.0f} MOhm, {g_sc*1e9:.0f} nS')
print('OTA output conductance gm / A0 = 2 pi UGB CL / A0 for an effective load CL of:', ', '.join(f'{c*1e12:.2f} pF: {2*np.pi*UGB*c/A0*1e9:.0f} nS' for c in (0.15e-12, 0.3e-12, 0.6e-12)))
print(f'charge per pin and cycle at a plate step of 0.5 V: {c_ser*0.5*1e15:.1f} fC, sim dither on minus off {(SIM["q_pin"][1]-SIM["q_pin"][0])*1e15:.1f} fC')
print(f'input gate area on / off ratio {(WIN+WAUX)/WIN:.2f}, sim charge ratio {SIM["q_pin"][1]/SIM["q_pin"][0]:.2f}')

# the precharge jump: vtail rises by about 0.8 V (README), a floating gate follows through Cgs
b2 = 2
cgs2 = abs(ext(nm, 'CGS', L, 0.35, 0.4, 0.4, WA[b2]))
cf2 = NCAP['2'] * CF_MODEL
cg2 = abs(ext(nm, 'CGG', L, 0.35, 0.4, 0.4, WA[b2]))
jump = 0.8 * cgs2 / (cgs2 + cf2 + cg2)
print(f'floating through the vtail jump: Cgs {cgs2*1e15:.1f} fF against Cf {cf2*1e15:.0f} fF, gate moves {jump*1e3:.0f} mV, {jump/(3*DV_STEP)*100:.0f} % of the 100 % tap')

flying cap unit 40.1 fF model, 28.1 fF estimated real
bit  Cf model  Cf real   Cgg aux   alpha model   alpha real
  0      40.1     28.1       7.4        0.843        0.790
  1      80.2     56.1      14.9        0.843        0.790
  2     160.4    112.3      29.8        0.843        0.790
A at s[0] with alpha model 85.2 mV, with alpha real 87.2 mV, sim 76.5 mV
kT/C of the held gate of bit 0: 0.38 mV rms, referred to the input 0.32 mV
series capacitance on each vi pin 44.0 fF, a switched-capacitor resistor of 22 MOhm, 45 nS
OTA output conductance gm / A0 = 2 pi UGB CL / A0 for an effective load CL of: 0.15 pF: 13 nS, 0.30 pF: 25 nS, 0.60 pF: 50 nS
charge per pin and cycle at a plate step of 0.5 V: 22.0 fC, sim dither on minus off 29.4 fC
input gate area on / off ratio 1.66, sim charge ratio 1.64
floating through the vtail jump: Cgs 5.4 fF against Cf 160 fF, gate moves 23 mV, 25 % of the 100 % tap


### Timing of the dither network

The delay between the clock edge and the gate and plate switching comes from `RDLY` (118 kOhm) into the gate of `MCDLY` (87 fF), 0.69 R C = 7.1 ns, which with the two delay cells is the 7 ns of the README. It has to exceed the 1 ns of the vtail jump and fit in the half period.

The plate switches (21u N and 21u P at L 0.13) have a worst-case Ron of 148 ohm over the plate voltage, 42 ps on the 281 fF of all plates. The gates reach the ladder through the tap resistance of 139 to 143 kOhm, so the bit 2 gate (the largest capacitance) settles with tau = 29 ns and has 17 tau in the high phase. The ladder is the slow part, not the switches. A held gate leaks through its two 0.5u switches with 18 pA (table, tt, at VDS = 0.4 V), which is 0.32 mV over a half period on the smallest bit. The table value is at the resolution floor of the sweep (it does not fall with L), so this is an upper bound at tt. Hot corners are not in the tables.

In [14]:
r_dly = RSQ * DT['RDLY']['l'] / DT['RDLY']['w']
c_dly = cgg('n', 1.0, DT['MCDLY']['w'] * DT['MCDLY']['n'])
print(f'RDLY {r_dly/1e3:.0f} kOhm, MCDLY gate {c_dly*1e15:.0f} fF, 0.69 RC = {0.69*r_dly*c_dly*1e9:.1f} ns (README: about 7 ns with the two delay cells)')

# plate switches: transmission gate worst case over the plate voltage
VS = np.arange(0.2, 1.31, 0.01)
wq = DT['MBCQN']['w'] * DT['MBCQN']['n']
gn = np.maximum(np.diag(nm.lookup('GDS', L=0.13, VGS=VDD - VS, VDS=0.01, VSB=VS)), 0) / nm['W']
gp = np.maximum(np.diag(pm.lookup('GDS', L=0.13, VGS=VS, VDS=0.01, VSB=VDD - VS)), 0) / pm['W']
ron_pl = 1 / (wq * (gn + gp)).clip(1e-12)
c_plate = sum(NCAP.values()) * CF_MODEL
print(f'plate switch {wq:g}u + {wq:g}u at L 0.13: worst Ron {ron_pl.max():.0f} ohm, plate node {c_plate*1e15:.0f} fF model, tau {ron_pl.max()*c_plate*1e12:.0f} ps')

# gate hold on the ladder: tap Thevenin resistance plus the mux and gate switches
RL1 = RSQ * DT['RL[7:0]']['l'] / DT['RL[7:0]']['w']
Rt = RSQ * DT['RTOP']['l'] / DT['RTOP']['w']
r_th = [(Rt + j * RL1) * (Rt + (8 - j) * RL1) / (2 * Rt + 8 * RL1) for j in range(9)]
r_sw = 2 * (ron_eff('n', 0.13, DT['MAMHN[3:0]']['w']) + ron_eff('p', 0.13, DT['MAMHP[3:0]']['w'])) / 2
tau_g = (max(r_th) + r_sw) * (NCAP['2'] * CF_MODEL + abs(ext(nm, 'CGG', L, 0.7, 0.5, VT, WA[2])))
print(f'tap Thevenin {min(r_th)/1e3:.0f} to {max(r_th)/1e3:.0f} kOhm, switches about {r_sw/1e3:.0f} kOhm, bit 2 gate tau {tau_g*1e9:.0f} ns, {(T_HALF-7e-9)/tau_g:.0f} tau in the high phase')

# off leakage of a gate switch pair (0.5u/0.13u), table value at tt, upper bound
ioff = (ext(nm, 'ID', 0.13, 0.0, 0.4, 0.0, DT['MGQHN[2:0]']['w']) + ext(pm, 'ID', 0.13, 0.0, 0.4, 0.0, DT['MGQHP[2:0]']['w']))
print(f'switch off current {ioff*1e12:.0f} pA, droop of bit 0 over a half period {ioff*T_HALF/(NCAP["0"]*CF_MODEL*CAP_REAL)*1e3:.2f} mV (tt, table floor)')

RDLY 118 kOhm, MCDLY gate 87 fF, 0.69 RC = 7.1 ns (README: about 7 ns with the two delay cells)


plate switch 21u + 21u at L 0.13: worst Ron 148 ohm, plate node 281 fF model, tau 42 ps


tap Thevenin 139 to 143 kOhm, switches about 7 kOhm, bit 2 gate tau 29 ns, 17 tau in the high phase
switch off current 18 pA, droop of bit 0 over a half period 0.32 mV (tt, table floor)


### Pad drivers and the preset gates

The pad path is `and2_1`, `inv_1` and `inv_2` into 80 fF, a last stage fanout of 12.4. The model gives 71 ps from the latch to the pad, the README 168 ps, so the model is 2.4 times low, the same direction as the clock path. The preset gates `nand2_2` and `nor2_2` add 8 and 10 ps in the model and up to 26 ps in the simulation. No analog size is derived for the drivers, they are library cells and the pad driver sees the IOPadOut4mA input as its only load.

The preset and reset currents are not sized quantities. Nothing in the preset path sets a current, every net has one driver and the current is clock gating plus leakage (46 nA disabled, 4.2 uA in reset with the clock running, README). The 195 uA of the earlier ratioed preset is what a driver fight costs, the present gate in front of the latch has none.

In [15]:
# pad driver: and2_1 -> inv_1 -> inv_2 -> 80 fF, from the latch output
t_pad = (stage('sg13cmos5l_and2_1', c_inv1) + stage('sg13cmos5l_inv_1', c_inv2) + stage('sg13cmos5l_inv_2', C_PAD))
print(f'input caps: inv_1 {c_inv1*1e15:.1f} fF, inv_2 {c_inv2*1e15:.1f} fF, pad load {C_PAD*1e15:.0f} fF, last stage fanout {C_PAD/c_inv2:.1f}')
print(f'latch to pad: model {t_pad*1e12:.0f} ps, sim (clock to do_q less clock to latch) {SIM["t_pad"]*1e12:.0f} ps')

# preset gate: one added stage per side in front of the SR latch
t_gp = stage('sg13cmos5l_nand2_2', c_nor, 2, 1)
t_gn = stage('sg13cmos5l_nor2_2', c_nor, 1, 2)
print(f'preset gates: nand2_2 {t_gp*1e12:.0f} ps, nor2_2 {t_gn*1e12:.0f} ps, sim: rising do_qi up to {SIM["d_gate"]*1e12:.0f} ps later, falling 0 to 18 ps earlier')
print(f'time left in the gap at the slow corner for a {V_RES*1e9:.0f} nV input: {(T_GAP - SLOW*t_dec(V_RES))*1e9:.2f} ns')

input caps: inv_1 3.2 fF, inv_2 6.5 fF, pad load 80 fF, last stage fanout 12.4
latch to pad: model 71 ps, sim (clock to do_q less clock to latch) 168 ps
preset gates: nand2_2 8 ps, nor2_2 10 ps, sim: rising do_qi up to 26 ps later, falling 0 to 18 ps earlier
time left in the gap at the slow corner for a 1 nV input: 0.75 ns


### Comparison with the drawing

In [16]:
rows = [
 ('input pair N0, N1 (W)',       W_MIN, WIN, 'smallest of the sweep that meets t and sigma', 'offset and kickback set it, Monte Carlo'),
 ('tail N2 (W)',                 2 * W_MIN, WTAIL, 'twice the input pair', 'follows the input pair'),
 ('aux pairs, total W per side', x_need * WIN, WAUX, f'A = {A_DITH*1e3:.0f} mV at the drawn ladder', ''),
]
print(f"{'group':30s} {'derived':>8s} {'drawn':>7s}   basis / verdict")
for n, a, b, why, v in rows:
    print(f'{n:30s} {a:8.1f} {b:7.1f}   {why} | ' + ('agrees' if abs(a - b) / b < 0.1 else 'drawn is larger, ' + v))
print(f"{'latch pairs, reset PMOS':30s} {'free':>8s} {'':>7s}   tau 69 to 103 ps over a factor 8 in width, Ron C 55 and 110 ps, drawn 2u / 6u / 6u, layout symmetry")
print(f"{'flying caps, units per side':30s} {'1:2:4':>8s} {sum(NCAP.values()):7d}   same ratio as the aux widths, agrees")
print(f"{'ladder current / uA':30s} {I_LAD*1e6:8.2f} {SIM['i_ladder']*1e6:7.1f}   drawn column is the sim")
print(f"{'dither A at s[0] / mV':30s} {amp(3*DV_STEP, x_gm, ALPHA_M)*1e3:8.1f} {SIM['A']*1e3:7.1f}   drawn column is the sim")

group                           derived   drawn   basis / verdict
input pair N0, N1 (W)               2.0    16.0   smallest of the sweep that meets t and sigma | drawn is larger, offset and kickback set it, Monte Carlo
tail N2 (W)                         4.0    32.0   twice the input pair | drawn is larger, follows the input pair
aux pairs, total W per side        10.5    10.5   A = 75 mV at the drawn ladder | agrees
latch pairs, reset PMOS            free           tau 69 to 103 ps over a factor 8 in width, Ron C 55 and 110 ps, drawn 2u / 6u / 6u, layout symmetry
flying caps, units per side       1:2:4       7   same ratio as the aux widths, agrees
ladder current / uA                2.63     2.6   drawn column is the sim
dither A at s[0] / mV              85.2    76.5   drawn column is the sim


The derivation reproduces the dither network and the timing it rests on: the aux width for 75 mV (10.5 um), the 1 : 2 : 4 weights, the ladder current (2.63 against 2.6 uA), the amplitude selects (33, 67, 100, 133 %) with 75.2 against 76.5 mV for alpha = 1, the 7.1 ns delay, the tau of the regeneration within 22 % and the kickback ratio (1.66 against 1.64).

Disagreements, none of which changes a drawn size:

- The input pair (16u) and the tail (32u) are 8 times wider than speed and noise need (2u and 4u). Offset and kickback set them and the Monte Carlo is the source.
- With the lookup gate capacitance the model amplitude at `s[0]` is 85.2 mV, 11 % above the simulated 76.5 mV. The simulation follows alpha = 1 better than the lookup.
- The decision time is 0.24 to 0.38 ns below the simulation, the pad path 2.4 times and the preset gates 2.5 times. The model has no wiring and no clock slew.
- The ss -40 C amplitude of 70.0 mV is 3 mV above the lower window edge of 67 mV (README, not derived).

Dither: the ideal dither adds about 0.5 dB of shaped error (the first version said 0.06 dB, from the wrong error scale). The switched-capacitor load of 45 nS per side that this notebook proposed as the cause of the 7.2 dB was rejected afterwards: a fit of the per-clock integrator step on the 16904-clock dither-on run shows no added leak (+1e-5 per clock against the 0.08 the 45 nS would give). The 7.2 dB comes from the fixed LFSR window after reset, see the dsm2 README.